In [16]:
#here we are cleaning up the csv results from the blast to make them unified in 1 file
#adding headers, remove those who are not pdb, saving the pubchem ids without blast results (for future use)

import os
import pandas as pd

# CONFIGURATION
INPUT_FOLDER = "/home/k_ensafitakaldani001_umb_edu/BLAST/FastaCSVs"
OUTPUT_FOLDER = "/home/k_ensafitakaldani001_umb_edu/BLAST/OutputCSV"
EXTENDED_OUTPUT_FOLDER = "/home/k_ensafitakaldani001_umb_edu/BLAST/output_extended"
LOG_FOLDER = "/home/k_ensafitakaldani001_umb_edu/BLAST"

EMPTY_LOG_FILE = os.path.join(LOG_FOLDER, "empty_files_log.txt")
FAILED_LOG_FILE = os.path.join(LOG_FOLDER, "failed_files_log.txt")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(EXTENDED_OUTPUT_FOLDER, exist_ok=True)

# HEADER DEFINITIONS (based on number of columns)
HEADERS_BY_COLUMNS = {
    7: [
        "query_id", "subject_id", "subject_description",
        "e_value", "bit_score", "percent_identity", "alignment_length"
    ],
    8: [
        "query_chain", "subject_id", "subject_chain",
        "subject_function", "e_value", "bit_score",
        "percent_identity", "alignment_length"
    ],
    9: [
        "query_chain", "subject_id", "subject_chain",
        "subject_name", "subject_description", "e_value",
        "bit_score", "percent_identity", "alignment_length"
    ]
}

# PROCESSING FUNCTION
def process_blast_csv(file_path, filename, empty_log, failed_log):
    try:
        if os.path.getsize(file_path) == 0:
            empty_log.write(f"{filename}\n")
            print(f"[SKIPPED] Empty file: {filename}")
            return

        df = pd.read_csv(file_path, header=None, engine="python", on_bad_lines="skip")
        col_count = df.shape[1]

        if col_count not in HEADERS_BY_COLUMNS:
            raise ValueError(f"Unhandled column count: {col_count}")

        df.columns = HEADERS_BY_COLUMNS[col_count]

        if col_count == 7:
            out_path = os.path.join(OUTPUT_FOLDER, filename.replace(".csv", "_with_headers.csv"))
        else:
            out_path = os.path.join(EXTENDED_OUTPUT_FOLDER, filename.replace(".csv", "_extended.csv"))

        df.to_csv(out_path, index=False)
        print(f"[OK] {filename} → {col_count}-column saved to {out_path}")

    except Exception as e:
        failed_log.write(f"{filename} — {e}\n")
        print(f"[ERROR] {filename} — {e}")

# MAIN SCRIPT
def main():
    with open(EMPTY_LOG_FILE, "w") as empty_log, open(FAILED_LOG_FILE, "w") as failed_log:
        for filename in os.listdir(INPUT_FOLDER):
            if filename.endswith(".csv"):
                file_path = os.path.join(INPUT_FOLDER, filename)
                process_blast_csv(file_path, filename, empty_log, failed_log)

    print(f"\n Empty files log saved to: {EMPTY_LOG_FILE}")
    print(f" Failed files log saved to: {FAILED_LOG_FILE}")

if __name__ == "__main__":
    main()


  Skipped empty file: 110D_pdb_hits.csv
  Skipped empty file: 127D_pdb_hits.csv
  Skipped empty file: 128D_pdb_hits.csv
  Skipped empty file: 152D_pdb_hits.csv
  Skipped empty file: 198D_pdb_hits.csv
  Processed: 1A29_pdb_hits.csv → 7-column
  Skipped empty file: 1BCU_pdb_hits.csv
  Skipped empty file: 1BKF_pdb_hits.csv
  Processed: 1C14_pdb_hits.csv → 8-column (extended)
  Skipped empty file: 1C3R_pdb_hits.csv
  Skipped empty file: 1C3S_pdb_hits.csv
  Processed: 1CET_pdb_hits.csv → 8-column (extended)
  Processed: 1CTR_pdb_hits.csv → 7-column
  Skipped empty file: 1D10_pdb_hits.csv
  Skipped empty file: 1D11_pdb_hits.csv
  Skipped empty file: 1D33_pdb_hits.csv
  Skipped empty file: 1D38_pdb_hits.csv
  Skipped empty file: 1D43_pdb_hits.csv
  Skipped empty file: 1D44_pdb_hits.csv
  Skipped empty file: 1D45_pdb_hits.csv
  Skipped empty file: 1D46_pdb_hits.csv
  Skipped empty file: 1D64_pdb_hits.csv
  Skipped empty file: 1D67_pdb_hits.csv
  Processed: 1D7O_pdb_hits.csv → 8-column (extende

In [18]:
#further cleanups for all the results by adding new columns to make the ultimate dataset

import os
import pandas as pd
import string

# CONFIGURATION
BLAST_DIR = "/home/k_ensafitakaldani001_umb_edu/BLAST"
OUTPUT_FOLDER = os.path.join(BLAST_DIR, "OutputCSV")

COMBINED_7_PATH = os.path.join(BLAST_DIR, "combined_7_columns.csv")
COMBINED_8_PATH = os.path.join(BLAST_DIR, "combined_8_columns.csv")
COMBINED_9_PATH = os.path.join(BLAST_DIR, "combined_9_columns.csv")
BLAST_PDB_RESULT = os.path.join(BLAST_DIR, "BlastPDBResult.csv")
FINAL_RESULT_PATH = os.path.join(BLAST_DIR, "target_malaria_blast_results.csv")

# COMBINE FILES BY COLUMN COUNT
def combine_blast_outputs():
    master_7 = pd.DataFrame()
    master_8 = pd.DataFrame()
    master_9 = pd.DataFrame()

    for filename in os.listdir(OUTPUT_FOLDER):
        if filename.endswith("_with_headers.csv") or filename.endswith("_extended.csv"):
            file_path = os.path.join(OUTPUT_FOLDER, filename)
            try:
                df = pd.read_csv(file_path).fillna("NaN")
                col_count = df.shape[1]

                if col_count == 7:
                    master_7 = pd.concat([master_7, df], ignore_index=True)
                    print(f"Appended to 7-column group: {filename}")
                elif col_count == 8:
                    master_8 = pd.concat([master_8, df], ignore_index=True)
                    print(f"Appended to 8-column group: {filename}")
                elif col_count == 9:
                    master_9 = pd.concat([master_9, df], ignore_index=True)
                    print(f"Appended to 9-column group: {filename}")
                else:
                    print(f"Skipped {filename}: unexpected column count ({col_count})")
            except Exception as e:
                print(f"Failed to process {filename} — {e}")

    master_7.to_csv(COMBINED_7_PATH, index=False)
    master_8.to_csv(COMBINED_8_PATH, index=False)
    master_9.to_csv(COMBINED_9_PATH, index=False)

    print(f"Combined 7-column CSV saved to: {COMBINED_7_PATH}")
    print(f"Combined 8-column CSV saved to: {COMBINED_8_PATH}")
    print(f"Combined 9-column CSV saved to: {COMBINED_9_PATH}")

# FILTER AND TRANSFORM PDB RESULTS
def filter_and_transform_blast_pdb():
    df7 = pd.read_csv(COMBINED_7_PATH)
    df8 = pd.read_csv(COMBINED_8_PATH)

    # Split subject_id by '|' and filter for 'pdb' entries
    df7['subject_id'] = df7['subject_id'].astype(str).apply(lambda x: x.split('|'))
    df8['subject_id'] = df8['subject_id'].astype(str).apply(lambda x: x.split('|'))

    df7 = df7[df7['subject_id'].apply(lambda x: 'pdb' in x)].copy()
    df8 = df8[df8['subject_id'].apply(lambda x: 'pdb' in x)].copy()

    df8.to_csv(BLAST_PDB_RESULT, index=False)
    print(f"Filtered PDB blast results saved to: {BLAST_PDB_RESULT}")

# FINAL CLEANUP AND FORMATTING
def finalize_blast_result():
    df = pd.read_csv(BLAST_PDB_RESULT)

    # Split query_chain
    df['query_chain'] = df['query_chain'].astype(str).apply(lambda x: x.split('|'))

    # Extract identifiers
    df['target_name'] = df['query_chain'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
    df['malaria_name'] = df['subject_id'].astype(str).str.strip("[]").str.replace("'", "").str.split(',').apply(
        lambda x: x[1].strip() if len(x) > 1 else None
    )

    # Extract numeric suffix after underscore for target_id
    df['target_id'] = df['target_name'].astype(str).apply(lambda x: x.split('_')[-1] if '_' in x else None)

    # Truncate target_name
    df['target_name'] = df['target_name'].astype(str).apply(lambda x: x[:4])

    # Extract last character of subject_chain
    df['malaria_id'] = df['subject_chain'].astype(str).apply(lambda x: x.strip()[-1] if len(x.strip()) > 0 else None)

    # Map numeric ID to alphabet chain (1 → A, 2 → B, ...)
    chain_id_map = {str(i + 1): letter for i, letter in enumerate(string.ascii_uppercase)}
    df['target_id'] = df['target_id'].map(chain_id_map)

    # Select relevant columns
    selected_columns = [
        'target_name', 'malaria_name', 'target_id', 'malaria_id',
        'e_value', 'bit_score', 'percent_identity', 'alignment_length'
    ]
    new_df = df[selected_columns].copy()
    new_df.to_csv(FINAL_RESULT_PATH, index=False)
    print(f"Final cleaned dataset saved to: {FINAL_RESULT_PATH}")

# MAIN
if __name__ == "__main__":
    combine_blast_outputs()
    filter_and_transform_blast_pdb()
    finalize_blast_result()


 Appended to 7-column group: 1A29_pdb_hits_with_headers.csv
 Appended to 8-column group: 1C14_pdb_hits_with_headers.csv
 Appended to 8-column group: 1CET_pdb_hits_with_headers.csv
 Appended to 7-column group: 1CTR_pdb_hits_with_headers.csv
 Appended to 8-column group: 1D7O_pdb_hits_with_headers.csv
 Appended to 8-column group: 1D8A_pdb_hits_with_headers.csv
 Appended to 8-column group: 1J3I_pdb_hits_with_headers.csv
 Appended to 7-column group: 1J3J_pdb_hits_with_headers.csv
 Appended to 8-column group: 1J3K_pdb_hits_with_headers.csv
 Appended to 7-column group: 1K73_pdb_hits_with_headers.csv
 Appended to 7-column group: 1LIN_pdb_hits_with_headers.csv
 Appended to 8-column group: 1NHG_pdb_hits_with_headers.csv
 Appended to 8-column group: 1P45_pdb_hits_with_headers.csv
 Appended to 8-column group: 1Q7Y_pdb_hits_with_headers.csv
 Appended to 8-column group: 1QG6_pdb_hits_with_headers.csv
 Appended to 8-column group: 1QSG_pdb_hits_with_headers.csv
 Appended to 7-column group: 1SA4_pdb_hi